# Task - 1: Data Exploration and Preprocessing

## Importing required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import scipy.stats as st
import nbformat

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None) # for showing all columns
pd.set_option('display.expand_frame_repr', False) # to prevent column header wrapping
pd.set_option('display.float_format', '{:.3f}'.format) # Float precision

sns.set_style("whitegrid")

## Loading data

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
from pathlib import Path

base_path = Path.cwd().parents[1]  # go up two levels
file_path = base_path / "Restaurant_Data" / "Dataset.csv"
df = pd.read_csv(file_path)

## Initial Inspection

In [ ]:
df.head(10)

In [ ]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
df.size

**Observation:**
- There are total 9551 rows and 21 columns in the give dataset.

In [ ]:
df.info()

**Observation:**
- There are total 8 numerical columns and 13 categorical columns in the given dataset.
- Numerical columns do not require any conversion.
- In further analysis, we may convert the follwoing 4 categorical columns into boolean data type if needed:
  1.  ***Has Table booking***
  2.  ***Has Online delivery***
  3.  ***Is delivering now***
  4.  ***Switch to order menu***
- As we can see, some of the columns in the dataset contains some null values which demands further inspection.

In [ ]:
# missing value inspection
df.isnull().sum()

In [ ]:
# visualization
plt.figure(figsize=(20,4))
bars = plt.bar(x=df.isnull().sum().index,
       height=df.isnull().sum().values,
       edgecolor='black',
       width=0.4,
       color='red')
plt.bar_label(bars,
              labels=df.isnull().sum().values,
              label_type='edge')
plt.xlabel('Name of the Features')
plt.xticks(rotation=45)
plt.ylabel('Missing Value Count')
plt.title('VISUALIZATION OF MISSING VALUES USING BAR GRAPH')
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "missing_value_analysis.png"
plt.savefig(file_path, dpi=150, bbox_inches="tight")
plt.show()

**Observation:**
- Only one column i.e. "Cuisines" contains 9 missings values which needs to be dealt with accordingly.

In [ ]:
df[df['Cuisines'].isnull()].reset_index(drop=True)

In [ ]:
result = str(np.round(df['Cuisines'].isnull().sum()/len(df)*100,2))+"%"
print("% missing values in the Cuisines column: {}".format(result))

### Missing value handling strategies:
- Dropping the rows with missing values
- Missing value Imputation
  - Forwardfill or Backwardfill
  - Mode imputation
  - We can create a separate category - "missing" so that ML model can analyze the pattern.
  - We can create a separate column - "Cuisines available" and fill that column with "True" for Cuisines not null and "False" for Cuisines null.

* As the missings values in the column is very small ~ 0.1%, so we can drop those rows with missing values.

In [ ]:
df.dropna(axis=0, subset=["Cuisines"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
# verifying
df[df['Cuisines'].isnull()].reset_index(drop=True)

In [ ]:
# duplicate rows inspection
df[df.duplicated(keep=False)]

**Observation:**
- There are no duplicated rows present in the given dataset.

## analyzing the distribution of the target variable - "Aggregate rating"

In [ ]:
# Helper function to plot histograms based on the
# format of the `sessions` histogram
def histogrammer(column_str, median_text=True, mean_text=True, **kwargs):    # **kwargs = any keyword arguments
                                                             # from the sns.histplot() function
    
    plt.figure(figsize=(20,3))

    if (df[column_str].dtypes != 'str') and (df[column_str].dtypes != 'object'):
        median=round(df[column_str].median(), 1)
        mean=round(df[column_str].mean(), 1)
        skewness=round(df[column_str].skew(), 1)
        kurtosis=round(st.kurtosis(df[column_str], fisher=False), 1)

        ax = sns.histplot(x=df[column_str], **kwargs)            # Plot the histogram
        plt.axvline(median, color='red', linestyle='--')         # Plot the median line
        plt.axvline(mean, color='green', linestyle='--')
        
        if (median_text==True) and (mean_text==True):                                    # Add median text unless set to False
            print("Skewness: {}\nKurtosis: {}".format(skewness, kurtosis))
            ax.text(0.25, 0.85, f'median={median}', color='red',
                ha='left', va='top', transform=ax.transAxes)
            ax.text(0.25, 0.95, f'mean={mean}', color='green',
                ha='left', va='top', transform=ax.transAxes)
        else:
            print('Median:', median)
            print('Mean:', mean)
    
    else:
        mode = df[column_str].mode()[0]
        print('Mode:', mode)
        sns.histplot(x=df[column_str], **kwargs)
        plt.xticks(rotation=90)
    
    plt.title(f'{column_str} histogram');

In [ ]:
histogrammer(column_str="Aggregate rating")

**Observation:**
- As we can see, there is a huge class imbalance in the aggregate rating column.

In [ ]:
df['Aggregate rating'].nunique()

In [ ]:
df['Aggregate rating'].unique()

In [ ]:
df['Aggregate rating'].value_counts()

In [ ]:
df['Aggregate rating'].value_counts(normalize=True)*100

In [ ]:
rating_text_list = list(df['Rating text'].value_counts().index)
rating_text_list

In [ ]:
for text in rating_text_list:
    print(f"_________________{text}_________________")
    print("Category wise rating count:")
    print(df[df['Rating text']==text]['Aggregate rating'].value_counts())
    print("="*35)
    print("\n")

In [ ]:
from utils.rating_histogram import histo

histo(df)

In [ ]:
df_rating = df[['Aggregate rating', 'Rating text', 'Votes']]

df_rating.head()

In [ ]:
df_rating[df_rating['Rating text']=="Not rated"]['Votes'].value_counts()

**Observation:**
- The zero / "Not Rated" problem is confirmed.
- Exactly 2,148 restaurants (22.5%) have a rating of 0.0, and the Rating text column labels them "Not rated" — these are definitely missing values, not true zero ratings.
- Crucially, 1,054 out of 2148 of these "Not rated" entries actually have non-zero votes, meaning some users voted but the platform never computed an aggregate. This makes them a distinct data quality issue.

In [ ]:
print("________________________Count________________________")
print(df['Rating text'].value_counts())
print("="*35)
print("\n")
print("________________________% of all________________________")
print(df['Rating text'].value_counts(normalize=True)*100)
print("="*35)
print("\n")
print("________________________% of only rated________________________")
temp_df = df[df['Rating text'] != "Not rated"]
print(df['Rating text'].value_counts()/len(temp_df)*100)

**Observation:**
- The "Average" class alone makes up 50.5% of all rated entries — a classic majority class dominance problem.
- "Excellent" is underrepresented at just 4.1%, with a 12.4:1 imbalance ratio between Average and Excellent.

In [ ]:
df.columns

In [ ]:
df_cat = df[['Country Code', 'Has Table booking','Has Online delivery', 
            'Is delivering now', 'Switch to order menu','Price range']]
df_cat.rename(columns={
    'Has Table booking': 'Table Booking',
    'Has Online delivery': 'Online delivery',
    'Is delivering now': 'Delivering now',
    'Switch to order menu': 'Order menu'
}, inplace=True)

df_cat.head()

In [ ]:
for col in df_cat.columns:
    print(f"____________________{col}____________________")
    print(df_cat[col].value_counts())
    print("="*40)
    print('\n')

In [ ]:
mapping = {1:'Budget' ,2:'Mid' ,3:'Premium' ,4:'Luxury'}
df_cat['Price range'] = df_cat['Price range'].replace(mapping)
df_cat.head(10)

In [ ]:
for col in df_cat.columns:
    print(f"____________________{col}____________________")
    print(df_cat[col].value_counts(normalize=True)*100)
    print("="*40)
    print('\n')

In [ ]:
# Proportion of count data on categorical columns
plt.figure(figsize=(15, 10))
plt.suptitle('Univariate Analysis of Categorical Features', fontsize=20, fontweight='bold', alpha=0.8, y=1)
colors = sns.color_palette("pastel")
for i,col in enumerate(df_cat.columns, start=1):
    index = df_cat[col].value_counts().index
    values = df_cat[col].value_counts().values
    plt.subplot(2, 3, i)
    sns.barplot(x=index,
               y=values,
               palette=colors
               )
    plt.xlabel(col)
    plt.tight_layout()
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Univariate_analysis_categorical_features.png"
plt.savefig(file_path, dpi=150, bbox_inches="tight");

In [ ]:
df[df['Country Code']==1]['City'].nunique()

In [ ]:
df[df['Country Code']==1]['City'].unique()

In [ ]:
df[['Locality', 'Locality Verbose']].head()

In [ ]:
df[['Locality', 'Locality Verbose']].tail()

In [ ]:
df.loc[3,'Locality Verbose']

In [ ]:
df.loc[9540,'Locality Verbose']

**Observation:**
- Additional imbalances worth noting:
1. ***Price range***: Heavily skewed toward budget (46.5%) — luxury restaurants are underrepresented at just 6.1%, which can bias any price-rating correlation analysis.
2. ***Online delivery***: 74.3% "No" vs 25.7% "Yes" — a 3:1 imbalance.
3. ***Delivering now***: A huge class imbalance which when comapred with Online delivery shows that even if online delivery is available in some restaurants, currently they may not provide the delivery service.
4. ***Table booking***: 87.9% "No" vs 12.1% "Yes" — a 7:1 imbalance.
5. ***Geography***: 90.6% of entries are from country code 1 (India), making cross-country generalization unreliable.
6. ***Order Menu***: There is only one unqiue value entered throughout the whole column.

In [ ]:
base_path = Path.cwd().parents[1]  
file_path = base_path / "Notebooks" / "processed_data" / "Dataset_filtered.csv"

df.to_csv(file_path, index=False)

#### For dasboard view, kindly refer to the rating_dashboard.py file by running the command: ***streamlit run rating_dashboard.py***

# Task - 2: Descriptive Analysis

In [ ]:
df.describe()

**Observation:**
- For "Average Cost for two" and "Votes" columns we can observe huge value discrepency.
- For "Average Cost for two", 75% of values are within 700 range where maximum value shows 800000. We should investigate further and try to find out the cause of this data variation, whether it is a simple mistake or there is a specific valid reason for that.
- For "Votes", 75% of values are within 130 range where maximum value shows 10934 i.e. some restaurants have gained a huge number of votes. We should investiagte this further and try to find out the cause. Is there any biasness exist for some particular restaurants from customer side or are these votes intentionally manipulated for further customer attraction.

### **Average Cost for two**

In [ ]:
# Box plot
plt.figure(figsize=(30,2))
sns.boxplot(
    x=df['Average Cost for two'],
    flierprops=dict(
        marker='x',
        markeredgecolor='red',
        markersize=8,
        linestyle='none'
    )
)
plt.title('Average Cost for two box plot');

In [ ]:
histogrammer('Average Cost for two', median_text=True, mean_text=True, kde=True)

**Observation:**
- This parameter is showing Highly Leptokurtic (i.e. Kurtosis >> 3) distribution.

### **Votes**

In [ ]:
# Box plot
plt.figure(figsize=(15,2))
sns.boxplot(
    x=df['Votes'],
    flierprops=dict(
        marker='x',
        markeredgecolor='red',
        markersize=8,
        linestyle='none'
    )
)
plt.title('Votes box plot');

In [ ]:
histogrammer('Votes', median_text=True, mean_text=True, kde=True)

**Observation:**
- This parameter is also showing Leptokurtic (i.e. Kurtosis > 3) distribution.

In [ ]:
df_1 = df[['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Cuisines', 'Currency']]

df_1.head()

In [ ]:
cols = [col for col in df_1.columns if col!='Cuisines']
for col in cols:
    print(f"________________{col}________________")
    print(f"Unique value count: {df_1[col].nunique()}")
    print("="*40)
    print("\n")

**Observation:**
- As the length of the dataset (i.e. 9542) and the unique Restaurant ID count are same, so we can say that all of the avalilable Restaurant IDs are unique.
- (unique Restaurant Name count) < (unique Restaurant ID count), so we can say that there may be multiple restaurants with same name but unique Restaurant ID.
- (unique Currency count) < (unique Country Code count), so we can say that there are multiple countries with same curreny.

### **Restaurant ID & Restaurant Name**

In [ ]:
df_1['Restaurant Name'].value_counts()

### **Country Code & Currency**

In [ ]:
df_1.groupby(by=["Currency",'Country Code'])[['City']].aggregate('count').reset_index()

**Observation:**
- Country codes: 14, 37, 184, 216 all have currency as Dollars($)

In [ ]:
for col in ["Country Code", "Currency"]:
    print(f"_________________{col}_________________")
    print(df_1[col].value_counts())
    print()
    print(df_1[col].value_counts(normalize=True)*100)
    print("="*35)
    print("\n")

In [ ]:
histogrammer("Country Code", kde=True)

In [ ]:
histogrammer("Currency", discrete=True)

### **City**

In [ ]:
print(df_1["City"].value_counts())
print("="*35)
print(df_1["City"].value_counts(normalize=True)*100)

In [ ]:
sns.set_style("white")
histogrammer("City", kde=True, discrete=True)

In [ ]:
df_city = df_1.groupby(by=['City'])[['Restaurant ID']].aggregate('count')
df_city.sort_values(by='Restaurant ID', ascending=False)

In [ ]:
values = df_1["City"].value_counts().values
index = df_1["City"].value_counts().index

plt.figure(figsize=(30,10))
ax = sns.barplot(x=index, 
                 y=values,
                 palette='Spectral',
                 hue=index,
                 legend=False)


for container in ax.containers:
    ax.bar_label(container, padding=3, rotation=90)

plt.xticks(rotation=90)
plt.title('City barplot', fontsize=16);

**Observation:**
1. New Deslhi - 5473
2. Gurgaon - 1118
3. Noida - 1080
4. Faridabad - 251

These 4 cities have huge value counts. So, we can analyze the remaining cities ignoring this 4 for now.

In [ ]:
values = df_1["City"].value_counts().values
index = df_1["City"].value_counts().index

plt.figure(figsize=(30,10))
ax = sns.barplot(x=index[4:], 
                 y=values[4:],
                 palette='Spectral',
                 hue=index[4:],
                 legend=False)


for container in ax.containers:
    ax.bar_label(container, padding=3, rotation=90)

plt.xticks(rotation=90)
plt.title('City barplot', fontsize=16);

### **Cuisines**

In [ ]:
# Filtering unique Cuisines

unique_cuisines = [] # initializing an empty list
for data in df_1['Cuisines'].str.split(', '):
    unique_cuisines.extend(data)

print(set(unique_cuisines)) # Unique cusisines
print("Available unique cuisines: {}".format(len(set(unique_cuisines)))) # No. of unique cuisines

In [ ]:
len(unique_cuisines)

In [ ]:
df_cuisines = pd.DataFrame(data=unique_cuisines, columns=['available_cuisines'])
df_cuisines

In [ ]:
print(df_cuisines.value_counts().reset_index())
print()
display(df_cuisines.value_counts(normalize=True)*100)

In [ ]:
# visualizing top 20 cuisisnes
data = df_cuisines.value_counts().reset_index()
index = data['available_cuisines']
values = data['count']

plt.figure(figsize=(15,10))
ax = sns.barplot(x=index[:20], 
                 y=values[:20],
                 palette='Spectral',
                 hue=index[:20],
                 legend=False)


for container in ax.containers:
    ax.bar_label(container, padding=3, rotation=90)

plt.xticks(rotation=90)
plt.title('Top 20 cuisines barplot', fontsize=16)
base_path = Path.cwd().parents[1]
file_path = base_path / "Notebooks" / "reports" / "Top_20_cuisines_barplot.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight');

# Task - 3: **Geospatial Analysis**

In [ ]:
import nbformat
print(nbformat.__version__)

In [ ]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
fig = px.scatter_mapbox(
    df,
    lat='Latitude',
    lon='Longitude',
    hover_name='Restaurant Name',
    hover_data=['City', 'Cuisines', 'Aggregate rating'],
    color='Aggregate rating',
    color_continuous_scale='RdYlGn',
    size='Votes',
    zoom=2,
    mapbox_style='open-street-map',
    title='Restaurant Locations by Rating'
)

fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [ ]:
df2 = df.groupby(by=['Country Code', 'City'])[['Restaurant ID']].aggregate('count').sort_values(by='Restaurant ID', ascending=False)

with pd.option_context('display.max_rows', None):
    display(df2)

In [ ]:
# Correlation
corr = df[['Latitude', 'Longitude', 'Aggregate rating']].corr()
print("Correlation with Aggregate rating:")
display(corr.round(3))

# City-level stats
city_stats = (
    df.groupby('City')
    .agg(avg_rating=('Aggregate rating', 'mean'),
         count=('Aggregate rating', 'count'))
    .query('count >= 20')
    .sort_values('avg_rating', ascending=False)
    .head(20)
)

# Summary table
print("\nTop 20 cities by average rating (min 20 restaurants):")
print(city_stats.round(2).to_string())

In [ ]:
# Plot: Rating by City (bar chart)
fig, axes = plt.subplots(2, 1, figsize=(20, 20))
fig.suptitle('Restaurant Location vs Rating Analysis', fontsize=20, fontweight='bold', y=1.01)

colors=[]
for v in city_stats['avg_rating']:
    if v >= 4.4:
        colors.append('#1D9E75')
    elif (4.4 > v >= 4.3):
        colors.append('#EF9F27')
    else:
        colors.append('#E24B4A')

axes[0].barh(city_stats.index, city_stats['avg_rating'], color=colors, edgecolor='white')
axes[0].set_xlabel('Average Rating')
axes[0].set_title('Top 20 Cities by Avg Rating\n(min 20 restaurants)', fontsize=17)
axes[0].set_xlim(3.0, 4.7)
axes[0].axvline(x=df['Aggregate rating'].mean(), color='gray', linestyle='--',
                linewidth=1, label=f"Overall avg ({df['Aggregate rating'].mean():.2f})")
axes[0].legend(fontsize=9)

# Add value labels
for i, (val, cnt) in enumerate(zip(city_stats['avg_rating'], city_stats['count'])):
    axes[0].text(val + 0.02, i, f'{val:.2f}  (n={cnt})', va='center', fontsize=8)

# Plot: Latitude (x) vs Longitude (y), bubble size = Aggregate Rating
bubble_size = df['Aggregate rating'] ** 2.5 * 10  # scale up for visibility

sc = axes[1].scatter(df['Latitude'], df['Longitude'],
                     alpha=0.4, s=bubble_size, c=df['Aggregate rating'],
                     cmap='RdYlGn', vmin=1, vmax=5)

axes[1].set_xlabel('Latitude')
axes[1].set_ylabel('Longitude')
axes[1].set_title('Restaurant Locations — Bubble Size = Rating', fontsize=17)
plt.colorbar(sc, ax=axes[1], label='Aggregate Rating')

# Add a legend showing bubble sizes
for rating in [2, 3, 4, 5]:
    axes[1].scatter([], [], s=rating**2.5 * 10, c='gray', alpha=0.5, label=f'Rating {rating}')
axes[1].legend(title='Rating scale', fontsize=8, title_fontsize=8)

plt.tight_layout()
base_path = Path.cwd().parents[1]  # go up two levels
file_path = base_path / "Notebooks" / "reports" / "location_vs_rating.png"
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()

**Observation:**
- There is a weak correlation, but location alone doesn't drive ratings. The correlation coefficients are 0.00 (latitude) and −0.12 (longitude), meaning higher-longitude cities (further east) tend to score slightly lower — but the relationship is too weak to be predictive.
- The more interesting story is city-level differences:
  * International cities (Abu Dhabi, Ankara, Athens) cluster around 4.2–4.3, significantly above average. These likely have fewer but better-quality restaurants in the dataset.
  * Indian metro cities (New Delhi, Noida, Gurgaon, Faridabad, Chennai) average 3.1–3.3. They dominate the dataset in volume (~5,600 restaurants) which drags the average down.
  * This is largely a data composition effect — India has ~90% of all restaurants in this dataset, so the overall averages are skewed by them.